In [5]:
from prefect import flow, task
import requests
import pandas as pd
from pathlib import Path
import pandera as pa
from pandera import Column, DataFrameSchema, Check

from common.loaders import load_valid_country_ids


API_URL = "https://restcountries.com/v3.1/all?fields=cca3,continents"

# --- Esquema de validación ---
continents_schema = DataFrameSchema({
    "cca3": Column(
        str,
        checks=[
            Check.str_length(3, 3),  # código alfa-3
            Check.isin(load_valid_country_ids())  # debe existir en countries.csv
        ],
        nullable=False
    ),
    "continents": Column(
        str,
        checks=[Check.isin([
            "Africa", "Antarctica", "Asia", "Europe",
            "North America", "Oceania", "South America"
        ])],
        nullable=False
    )
})

@task
def extract_continents():
    response = requests.get(API_URL)
    response.raise_for_status()
    return response.json()

@task
def transform_continents(data):
    df = pd.DataFrame(data)
    # Explode por países con múltiples continentes (ej: RUS, TUR, AZE)
    df_exploded = df.explode("continents").reset_index(drop=True)
    return df_exploded

@task
def validate_continents(df: pd.DataFrame) -> pd.DataFrame:
    return continents_schema.validate(df)

@task
def load_continents(df: pd.DataFrame):
    file_path = Path("../stage/country_continents.csv")
    file_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(file_path, index=False, encoding="utf-8")
    print(f"Saved {len(df)} rows to {file_path}")

@flow(name="etl_continents_flow")
def etl_continents():
    data = extract_continents()
    df = transform_continents(data)
    df = validate_continents(df)
    load_continents(df)

if __name__ == "__main__":
    etl_continents()


C:\Users\Sebastian\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandera\_pandas_deprecated.py:149: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


03:35:22.987 | INFO    | Flow run 'violet-monkey' - Beginning flow run 'violet-monkey' for flow 'etl_continents_flow'

03:35:23.969 | INFO    | Task run 'extract_continents-d03' - Finished in state Completed()

03:35:24.203 | INFO    | Task run 'transform_continents-298' - Finished in state Completed()

03:35:24.438 | INFO    | Task run 'validate_continents-886' - Finished in state Completed()

Saved 253 rows to ..\stage\country_continents.csv


03:35:24.659 | INFO    | Task run 'load_continents-935' - Finished in state Completed()

03:35:24.672 | INFO    | Flow run 'violet-monkey' - Finished in state Completed()